# Data Wrangling Pipeline for Corporate Bankruptcy (Poland & Taiwan): From Raw JSON to Analysis-Ready DataFrame

1. Contoh JSON valid

- json.loads(good_json) mengubah string JSON menjadi dict Python.
- obj["features"]["assets"] mengambil nilai bersarang (hasil: 45000.0).
- Tujuannya adalah pemanasan: memahami bahwa null menjadi None, dan JSON bisa bersarang.

In [1]:
import json

good_json = """
{
  "company_id": 123,
  "country": "PL",
  "features": {
    "assets": 45000.0,
    "net_profit": -1200.5
  },
  "bankrupt": 1,
  "tags": ["manufacturing", "export"],
  "note": null
}
"""

obj = json.loads(good_json)
type(obj), obj["features"]["assets"]

(dict, 45000.0)

2. Contoh JSON tidak valid

- bad_json sengaja dibuat salah: kutip tunggal, koma berlebih (trailing comma), dan True (huruf besar, seharusnya true).
- try/except json.JSONDecodeError menangkap error dan mencetak pesan, baris, serta kolom kesalahan.

In [2]:
bad_json = """
{
'company_id': 123,
"features": {"assets": 45000.0, "net_profit": -1200.5,},
"bankrupt": True
}
"""

try:
    json.loads(bad_json)
except json.JSONDecodeError as err:
    print("JSONDecodeError:", err.msg)
    print("Line:", err.lineno, "Column:", err.colno)

JSONDecodeError: Expecting property name enclosed in double quotes
Line: 3 Column: 1


3. Import library

- gzip untuk membaca file .gz, json untuk parsing, Path untuk path file lintas OS, pandas untuk tabel data.
- pd.set_option mengatur agar tampilan DataFrame menampilkan hingga 120 kolom.

In [3]:
from __future__ import annotations

import gzip
import json
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

4. Menemukan dan memvalidasi file

- data_dir.glob("*.json*") mengumpulkan semua file .json / .json.gz di folder data.
- assert len(paths) == 4 memastikan ada tepat 4 file.
- File dipisah menjadi poland_paths dan taiwan_paths berdasarkan nama file, lalu divalidasi lagi dengan assert.

In [4]:
data_dir = Path("data")
paths = sorted(data_dir.glob("*.json*"))
paths

[WindowsPath('data/poland-bankruptcy-data-2009-mvp-features.json.gz'),
 WindowsPath('data/poland-bankruptcy-data-2009.json.gz'),
 WindowsPath('data/taiwan-bankruptcy-data-test-features.json.gz'),
 WindowsPath('data/taiwan-bankruptcy-data.json.gz')]

In [5]:
assert len(paths) == 4, "Inside data folder you should have four .json/.json.gz files."

In [6]:
poland_paths = sorted([p for p in paths if "poland" in p.name.lower()])
taiwan_paths = sorted([p for p in paths if "taiwan" in p.name.lower()])

poland_paths, taiwan_paths

([WindowsPath('data/poland-bankruptcy-data-2009-mvp-features.json.gz'),
  WindowsPath('data/poland-bankruptcy-data-2009.json.gz')],
 [WindowsPath('data/taiwan-bankruptcy-data-test-features.json.gz'),
  WindowsPath('data/taiwan-bankruptcy-data.json.gz')])

In [7]:
assert len(poland_paths) == 2, "Expected 2 Poland files"
assert len(taiwan_paths) == 2, "Expected 2 Taiwan files"

assert all(["poland" in str(p).lower() for p in poland_paths]), "Expected 2 Poland files"
assert all(["taiwan" in str(p).lower() for p in taiwan_paths]), "Expected 2 Taiwan files"

5. load_json()

- Membaca file JSON biasa atau terkompresi gzip (dicek lewat suffix == ".gz").

In [8]:
def load_json(path: str | Path) -> Any:
    """Load a JSON file that may be gzip-compressed.

    Parameters
    ----------
    path
        Path to a ``.json`` or ``.json.gz`` file.

    Returns
    -------
    Any
        Parsed JSON (often a dict, sometimes a list).
    """
    p = Path(path)  # use Path

    if p.suffix == ".gz":
        with gzip.open(p, "rt", encoding="utf-8") as f:
            return json.load(f)

    with p.open(p, "rt", encoding="utf-8") as f:
        return json.load(f)

6. Inspeksi struktur

- Memuat file pertama dan mengecek tipe root (dict) dan kunci tingkat atas: schema, data, metadata.

In [9]:
payload0 = load_json(paths[0])

print("Root type:", type(payload0))
if isinstance(payload0, dict):
    print("Top-level keys:", list(payload0.keys()))

assert payload0 is not None

Root type: <class 'dict'>
Top-level keys: ['schema', 'data', 'metadata']


7. extract_records()

- Mengambil daftar record dari kunci data atau observations.
- Memvalidasi tipe (harus dict → list → list of dict), jika tidak, melempar KeyError/TypeError dengan pesan jelas.
- Cell 11 mengecek hasilnya dan menampilkan 10 nama kolom pertama (company_id, feat_1, dst.).

In [10]:
def extract_records(payload: Any) -> list[dict[str, Any]]:
    """
    Extract the list of record dictionaries from the dataset payload.

    Parameters
    ----------
    payload
        Parsed JSON payload (expected: dict with records under a known key).

    Returns
    -------
    list of dict
        The dataset records.

    Raises
    ------
    KeyError
        If neither ``data`` nor ``observations`` is present.
    TypeError
        If the record container exists but is not a list of dicts.
    """
    if not isinstance(payload, dict):
        raise TypeError("Expected the JSON payload to be a dict.")

    if "data" in payload:
        records = payload["data"]
    elif "observations" in payload:
        records = payload["observations"]
    else:
        raise KeyError(
            "Could not find records. Expected key 'data' or 'observations'. "
            f"Found keys: {list(payload.keys())}"
        )

    if not isinstance(records, list):
        raise TypeError("Expected the records container to be a list.")

    if records and not isinstance(records[0], dict):
        raise TypeError("Expected records to be a list of dicts.")

    return records

In [11]:
records0 = extract_records(payload0)

assert isinstance(records0, list)
assert len(records0) > 0
assert isinstance(records0[0], dict)

list(records0[0].keys())[:10]

['company_id',
 'feat_1',
 'feat_2',
 'feat_3',
 'feat_4',
 'feat_5',
 'feat_6',
 'feat_7',
 'feat_8',
 'feat_9']

7. records_to_frame()

- Mengubah list of dict menjadi DataFrame.
- Jika ada nilai bersarang (dict/list), memakai pd.json_normalize; jika tidak, DataFrame.from_records.
- Cell 13 menampilkan 5 baris pertama: 64 fitur rasio keuangan, dengan banyak nilai NaN.

In [12]:
def records_to_frame(records: list[dict[str, Any]]) -> pd.DataFrame:
    """
    Convert a list of record dicts into a DataFrame.

    Parameters
    ----------
    records
        List of dictionaries, one per observation.

    Returns
    -------
    pandas.DataFrame
        DataFrame created from records.
    """
    has_nested = any(
        isinstance(v, (dict, list))
        for r in records
        for v in r.values()
    )
    if has_nested:
        return pd.json_normalize(records)
    return pd.DataFrame.from_records(records)

In [13]:
df_raw0 = records_to_frame(records0)

assert isinstance(df_raw0, pd.DataFrame)
assert df_raw0.shape[0] > 0
assert df_raw0.shape[1] > 0

df_raw0.head()

,company_id,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,feat_10,feat_11,feat_12,feat_13,feat_14,feat_15,feat_16,feat_17,feat_18,feat_19,feat_20,feat_21,feat_22,feat_23,feat_24,feat_25,feat_26,feat_27,feat_28,feat_29,feat_30,feat_31,feat_32,feat_33,feat_34,feat_35,feat_36,feat_37,feat_38,feat_39,feat_40,feat_41,feat_42,feat_43,feat_44,feat_45,feat_46,feat_47,feat_48,feat_49,feat_50,feat_51,feat_52,feat_53,feat_54,feat_55,feat_56,feat_57,feat_58,feat_59,feat_60,feat_61,feat_62,feat_63,feat_64
0,4,0.024526,0.43236,0.27546,1.78330,-10.105,0.56944,0.024526,1.30570,1.05090,0.56453,0.024526,0.069747,0.036812,0.024526,3299.40,0.110630,2.31290,0.024526,0.018876,86.317,0.62795,0.055446,0.018876,0.56944,0.564530,0.110630,0.44844,0.73869,4.2600,0.28624,0.018876,103.810,3.5161,0.12824,0.055446,1.30680,3.9624,0.645240,0.042673,0.17826,0.180500,0.042673,158.55,72.237,0.079819,0.90954,90.707,0.032141,0.024737,1.45040,0.35164,0.28440,1.51390,1.730300,5012.600,0.048398,0.043445,0.95160,0.14298,4.2286,5.0528,98.783,3.6950,3.4844
1,32,0.266780,0.41576,0.62014,3.89080,64.833,0.00000,0.332720,1.40520,1.71680,0.58424,0.335930,1.551000,0.201880,0.332720,437.84,0.833640,2.40520,0.332720,0.193800,78.204,0.70047,0.334180,0.155390,NaN,0.406660,0.675050,103.84000,3.75070,4.4496,0.22590,0.193880,56.947,6.4094,3.30710,0.341860,1.71680,NaN,0.584240,0.199130,0.16362,0.039818,0.194650,169.99,91.784,0.725250,2.17610,97.649,0.320300,0.186560,2.00760,0.21452,0.15602,3.53360,3.533600,17461.000,0.199130,0.456630,0.81067,0.00000,4.6673,3.9767,45.608,8.0030,10.3840
2,34,0.003892,0.85020,0.27146,1.37260,10.360,0.00000,0.003892,0.17620,2.09600,0.14980,0.011205,0.005342,0.001857,0.003892,79735.00,0.004578,1.17620,0.003892,0.001857,35.780,1.48500,0.011205,0.001857,NaN,0.012959,0.004578,1.53220,NaN,2.5627,0.27466,0.005346,127.260,2.8680,2.45770,0.006530,2.09600,6.5313,0.271460,0.003116,0.38482,2.529200,0.005346,125.32,89.539,0.018942,1.09060,35.892,0.011205,0.005346,1.17620,0.72854,0.34867,NaN,NaN,99.183,0.003116,0.025980,0.99815,0.81206,10.2010,4.0764,126.870,2.8770,NaN
3,36,0.044384,1.06430,-0.52980,0.43939,-158.400,-0.52553,0.044384,-0.14114,1.07410,-0.15021,0.044384,0.046965,0.065540,0.044384,3489.20,0.104610,0.93959,0.044384,0.026127,16.954,0.98457,0.125340,0.026127,-0.52553,-0.150210,0.104610,0.79254,-0.90601,4.0275,0.59029,0.026127,218.100,1.6735,0.11777,0.125340,1.82000,2.8202,-0.030954,0.073786,0.12617,0.181960,0.073786,63.60,46.646,0.562510,0.35589,18.210,0.058392,0.034374,0.39015,0.94504,0.59754,-0.25687,-0.052934,-5644.000,0.068995,-0.295480,0.93100,-0.79393,21.5290,7.8248,203.050,1.7975,2.9050
4,40,0.013141,0.38626,0.38047,2.24300,44.200,0.00000,0.013141,1.58890,0.88321,0.61374,0.015213,0.042933,0.055477,0.013141,2877.40,0.126850,2.58890,0.013141,0.014879,107.620,NaN,0.007733,0.014879,NaN,0.492290,0.126850,3.73230,1.21390,3.9275,0.19605,0.015318,111.670,3.2684,2.59000,-0.117220,0.88321,NaN,0.613740,-0.132720,0.70159,0.295380,0.008755,194.98,87.366,0.050464,1.39230,95.007,-0.028124,-0.031843,1.77740,0.30609,0.30596,1.95810,1.958100,3220.000,-0.132720,0.021412,0.98720,0.00000,3.3917,4.1778,126.500,2.8855,2.8178


8. normalize_columns()

- Menyeragamkan nama kolom: strip, huruf kecil, spasi/tanda hubung menjadi _, buang karakter non-alfanumerik, rapikan __ ganda.
- Cell 15 membuktikan bahwa nama kolom dataset ini sudah rapi (tidak berubah).

In [14]:
def normalize_columns(columns: list[str]) -> list[str]:
    """
    Normalize column names to a consistent snake_case style.

    Parameters
    ----------
    columns
        Original column names.

    Returns
    -------
    list of str
        Normalized column names.
    """
    out: list[str] = []
    for c in columns:
        c = str(c)
        c = c.strip()  # strip
        c = c.lower() # lower
        c = c.replace(" ", "_") # replace space by underscore
        c = c.replace("-", "_") # replace dash by underscore
        # check if ch is alpha numeric
        c = "".join(ch for ch in c if ch.isalnum() or ch == "_")
        while "__" in c:
            c = c.replace("__", "_")  # replace double underscore by single underscore
        out.append(c)
    return out

In [15]:
df_tmp0 = df_raw0.copy()
df_tmp0.columns = normalize_columns([str(c) for c in df_tmp0.columns])

assert df_tmp0.columns.equals(df_raw0.columns), (
    "Column names changed after normalization. "
    "This dataset may contain spaces/dashes/case differences that were "
    "standardized by `normalize_columns`."
)

9. clean_financials()

- Menormalkan nama kolom dan mengubah semua kolom menjadi numerik dengan pd.to_numeric(errors="coerce") (nilai yang tidak bisa dikonversi menjadi NaN).
- Shape sebelum dan sesudah dicek tetap sama.

In [16]:
def clean_financials(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean dataset for modeling (simple version).

    Parameters
    ----------
    df
        Raw DataFrame extracted from JSON.

    Returns
    -------
    pandas.DataFrame
        Cleaned DataFrame with normalized columns and numeric values where
        possible.
    """
    df2 = df.copy()
    df2.columns = normalize_columns([str(c) for c in df2.columns])

    for col in df2.columns:
        df2[col] = pd.to_numeric(df2[col], errors="coerce")

    return df2

In [17]:
df_clean0 = clean_financials(df_raw0)

assert isinstance(df_clean0, pd.DataFrame)
assert df_clean0.shape == df_raw0.shape
df_clean0.head()

,company_id,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,feat_10,feat_11,feat_12,feat_13,feat_14,feat_15,feat_16,feat_17,feat_18,feat_19,feat_20,feat_21,feat_22,feat_23,feat_24,feat_25,feat_26,feat_27,feat_28,feat_29,feat_30,feat_31,feat_32,feat_33,feat_34,feat_35,feat_36,feat_37,feat_38,feat_39,feat_40,feat_41,feat_42,feat_43,feat_44,feat_45,feat_46,feat_47,feat_48,feat_49,feat_50,feat_51,feat_52,feat_53,feat_54,feat_55,feat_56,feat_57,feat_58,feat_59,feat_60,feat_61,feat_62,feat_63,feat_64
0,4,0.024526,0.43236,0.27546,1.78330,-10.105,0.56944,0.024526,1.30570,1.05090,0.56453,0.024526,0.069747,0.036812,0.024526,3299.40,0.110630,2.31290,0.024526,0.018876,86.317,0.62795,0.055446,0.018876,0.56944,0.564530,0.110630,0.44844,0.73869,4.2600,0.28624,0.018876,103.810,3.5161,0.12824,0.055446,1.30680,3.9624,0.645240,0.042673,0.17826,0.180500,0.042673,158.55,72.237,0.079819,0.90954,90.707,0.032141,0.024737,1.45040,0.35164,0.28440,1.51390,1.730300,5012.600,0.048398,0.043445,0.95160,0.14298,4.2286,5.0528,98.783,3.6950,3.4844
1,32,0.266780,0.41576,0.62014,3.89080,64.833,0.00000,0.332720,1.40520,1.71680,0.58424,0.335930,1.551000,0.201880,0.332720,437.84,0.833640,2.40520,0.332720,0.193800,78.204,0.70047,0.334180,0.155390,NaN,0.406660,0.675050,103.84000,3.75070,4.4496,0.22590,0.193880,56.947,6.4094,3.30710,0.341860,1.71680,NaN,0.584240,0.199130,0.16362,0.039818,0.194650,169.99,91.784,0.725250,2.17610,97.649,0.320300,0.186560,2.00760,0.21452,0.15602,3.53360,3.533600,17461.000,0.199130,0.456630,0.81067,0.00000,4.6673,3.9767,45.608,8.0030,10.3840
2,34,0.003892,0.85020,0.27146,1.37260,10.360,0.00000,0.003892,0.17620,2.09600,0.14980,0.011205,0.005342,0.001857,0.003892,79735.00,0.004578,1.17620,0.003892,0.001857,35.780,1.48500,0.011205,0.001857,NaN,0.012959,0.004578,1.53220,NaN,2.5627,0.27466,0.005346,127.260,2.8680,2.45770,0.006530,2.09600,6.5313,0.271460,0.003116,0.38482,2.529200,0.005346,125.32,89.539,0.018942,1.09060,35.892,0.011205,0.005346,1.17620,0.72854,0.34867,NaN,NaN,99.183,0.003116,0.025980,0.99815,0.81206,10.2010,4.0764,126.870,2.8770,NaN
3,36,0.044384,1.06430,-0.52980,0.43939,-158.400,-0.52553,0.044384,-0.14114,1.07410,-0.15021,0.044384,0.046965,0.065540,0.044384,3489.20,0.104610,0.93959,0.044384,0.026127,16.954,0.98457,0.125340,0.026127,-0.52553,-0.150210,0.104610,0.79254,-0.90601,4.0275,0.59029,0.026127,218.100,1.6735,0.11777,0.125340,1.82000,2.8202,-0.030954,0.073786,0.12617,0.181960,0.073786,63.60,46.646,0.562510,0.35589,18.210,0.058392,0.034374,0.39015,0.94504,0.59754,-0.25687,-0.052934,-5644.000,0.068995,-0.295480,0.93100,-0.79393,21.5290,7.8248,203.050,1.7975,2.9050
4,40,0.013141,0.38626,0.38047,2.24300,44.200,0.00000,0.013141,1.58890,0.88321,0.61374,0.015213,0.042933,0.055477,0.013141,2877.40,0.126850,2.58890,0.013141,0.014879,107.620,NaN,0.007733,0.014879,NaN,0.492290,0.126850,3.73230,1.21390,3.9275,0.19605,0.015318,111.670,3.2684,2.59000,-0.117220,0.88321,NaN,0.613740,-0.132720,0.70159,0.295380,0.008755,194.98,87.366,0.050464,1.39230,95.007,-0.028124,-0.031843,1.77740,0.30609,0.30596,1.95810,1.958100,3220.000,-0.132720,0.021412,0.98720,0.00000,3.3917,4.1778,126.500,2.8855,2.8178


10. wrangle()

- Fungsi pembungkus satu langkah: load_json → extract_records → records_to_frame → clean_financials.
- Diuji pada file pertama.

In [18]:
def wrangle(path: str | Path) -> pd.DataFrame:
    """
    Load + transform a bankruptcy dataset into a clean DataFrame.

    Parameters
    ----------
    path
        Path to a ``.json`` or ``.json.gz`` file.

    Returns
    -------
    pandas.DataFrame
        Cleaned DataFrame ready for analysis.
    """
    payload = load_json(path)
    records = extract_records(payload)
    df_raw = records_to_frame(records)
    return clean_financials(df_raw)

In [19]:
df_w0 = wrangle(paths[0])

assert isinstance(df_w0, pd.DataFrame)
assert df_w0.shape[0] > 0
assert df_w0.select_dtypes(include=["number"]).shape[1] > 0

11. Memuat semua dataset

- datasets_path memetakan 4 dataset berdasarkan nama (ada kata “features” atau tidak).
- frames menyimpan hasil wrangle tiap file. Ukurannya:
    - poland_features: 526 × 65
    - poland_full: 9.977 × 66
    - taiwan_features: 682 × 96
    - taiwan_full: 6.137 × 97

In [20]:
datasets_path: dict[str, Path] = {
    "poland_features": next(
        p for p in poland_paths if "features" in p.name.lower()
    ),
    "poland_full": next(
        p for p in poland_paths if "features" not in p.name.lower()
    ),
    "taiwan_features": next(
        p for p in taiwan_paths if "features" in p.name.lower()
    ),
    "taiwan_full": next(
        p for p in taiwan_paths if "features" not in p.name.lower()
    ),
}

datasets_path

{'poland_features': WindowsPath('data/poland-bankruptcy-data-2009-mvp-features.json.gz'),
 'poland_full': WindowsPath('data/poland-bankruptcy-data-2009.json.gz'),
 'taiwan_features': WindowsPath('data/taiwan-bankruptcy-data-test-features.json.gz'),
 'taiwan_full': WindowsPath('data/taiwan-bankruptcy-data.json.gz')}

In [21]:
frames: dict[str, pd.DataFrame] = {
    name: wrangle(path) for name, path in datasets_path.items()
}

{k: v.shape for k, v in frames.items()}

{'poland_features': (526, 65),
 'poland_full': (9977, 66),
 'taiwan_features': (682, 96),
 'taiwan_full': (6137, 97)}

12. Validasi skema

- Memastikan kolom ID ada (company_id untuk Polandia, id untuk Taiwan).
- Dataset *_full punya kolom target bankrupt, dataset *_features tidak.
- expected_feature_set() membuat set feat_1..feat_n (64 fitur Polandia, 95 fitur Taiwan) dan mengecek tidak ada yang hilang.

In [22]:
assert "company_id" in frames["poland_features"].columns
assert "company_id" in frames["poland_full"].columns
assert "id" in frames["taiwan_features"].columns
assert "id" in frames["taiwan_full"].columns

assert "bankrupt" not in frames["poland_features"].columns
assert "bankrupt" in frames["poland_full"].columns
assert "bankrupt" not in frames["taiwan_features"].columns
assert "bankrupt" in frames["taiwan_full"].columns

In [23]:
def expected_feature_set(n_feats: int) -> set[str]:
    """Return expected feature columns: feat_1..feat_n."""
    return {f"feat_{i}" for i in range(1, n_feats + 1)}

poland_feats = expected_feature_set(64)
taiwan_feats = expected_feature_set(95)

checks = [
    ("poland_features", "company_id", poland_feats, False),
    ("poland_full", "company_id", poland_feats, True),
    ("taiwan_features", "id", taiwan_feats, False),
    ("taiwan_full", "id", taiwan_feats, True),
]

for name, id_col, feat_set, has_target in checks:
    df_ = frames[name]
    cols = set(df_.columns)

    assert id_col in cols, f"{name}: missing id column {id_col!r}"

    missing_feats = feat_set - cols
    assert not missing_feats, (
        f"{name}: missing features (sample): "
        f"{sorted(missing_feats)[:10]}"
    )

    if has_target:
        assert "bankrupt" in cols, f"{name}: expected 'bankrupt'"
    else:
        assert "bankrupt" not in cols, f"{name}: did not expect 'bankrupt'"

print("All column checks passed!")

All column checks passed!


13. imbalance_ratio()

- Menghitung rasio kelas mayoritas : minoritas pada kolom bankrupt (mengembalikan NaN jika hanya satu kelas, inf jika minoritas nol).
- Hasil: Polandia 20,36, Taiwan 31,30.

In [24]:
def imbalance_ratio(y: pd.Series) -> float:
    """Compute imbalance ratio: majority_count / minority_count."""
    s = y.dropna().astype(int)  # drop Nan from `y` and cast values to int
    counts = s.value_counts()  # counts of unique values from `s`

    if counts.shape[0] < 2:
        # If the size of `counts` is less than 2, the result is invalid,
        # as it should contain counts for both values (0 and 1).
        return float("nan")

    majority = counts.max()  # get the max from the `counts`
    minority = counts.min()  # get the min from the `counts`

    if minority == 0.0:
        # division by zero is invalid, so returns "inf"
        return float("inf")

    return majority / minority # calculate the division between majority and minority

In [25]:
labeled = {
    "poland_full": frames['poland_full'],  # get the "poland_full" from `frames`
    "taiwan_full": frames['taiwan_full'], # get the "taiwan_full" from `frames`
}

for name, df_ in labeled.items():
    vals = set(df_["bankrupt"].dropna().astype(int).unique())
    assert vals.issubset({0, 1}), f"{name} target not binary: {sorted(vals)}"

    ir = imbalance_ratio(df_["bankrupt"]) # get the imbalance ratio for "bankrupt" from `df_`
    print(f"{name}: imbalance ratio = {ir:.2f}")

poland_full: imbalance ratio = 20.36
taiwan_full: imbalance ratio = 31.30


Kesimpulan Singkat

- Pipeline data sudah rapi dan modular. 
- Fungsi-fungsi kecil (load_json, extract_records, records_to_frame, clean_financials, wrangle) digabung menjadi satu alur yang bisa dipakai ulang untuk keempat dataset.
- Validasi berlapis (assert, pengecekan skema, kolom ID, fitur, dan target) membuat kualitas data terjamin sebelum masuk ke tahap pemodelan.
- Data sangat tidak seimbang (imbalanced). 
- Perusahaan bangkrut hanya sekitar 1 dari 20 (Polandia) dan 1 dari 31 (Taiwan). 
- Akurasi saja akan menyesatkan, jadi tahap berikutnya sebaiknya memakai metrik seperti recall, precision, F1, PR-AUC, serta teknik penanganan seperti class weight, SMOTE, atau undersampling.
- Nilai hilang (NaN) cukup banyak pada fitur, sehingga perlu strategi imputasi sebelum modeling.